In [1]:
from datasets import load_dataset
from trl import RewardTrainer, RewardConfig
import wandb
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

wandb.init(project='mini-llm', name='RewardModel')

model_name = "./checkpoints/sft/sft-seed42/final"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=1, dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized", split="train_prefs"
).select(range(20000)).shuffle(seed=42)

/home/artmak/Desktop/Education/mini-LLM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/artmak/.netrc.
wandb: Currently logged in as: artmak (artmak-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: 500 encountered ({"errors":[{"message":"driver: bad connection","path":["upsertBucket"]}],"data":{"upsertBucket":null}}), retrying request
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 30358.61it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: ./checkpoints/sft/sft-seed42/final
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [2]:
val = dataset.select(range(1000))
train = dataset.select(range(1000, 20000))

def format_pair(example):
    rejected = tokenizer.apply_chat_template(example['rejected'], tokenize=False)
    chosen = tokenizer.apply_chat_template(example['chosen'], tokenize=False)
    return {'prompt': example['prompt'],
            'rejected': rejected,
            'chosen':chosen}

train = train.map(format_pair)
val = val.map(format_pair)

In [3]:
config = RewardConfig(
    output_dir="./checkpoints/rm",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    bf16=True,
    num_train_epochs=1,
    max_length=1024,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    center_rewards_coefficient=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=["wandb"],
    warmup_steps=50,
)

trainer = RewardTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train, eval_dataset=val,
    args=config,
)
trainer.train()

Filtering eval >1024 tokens: 100%|██████████| 1000/1000 [00:00<00:00, 46817.70 examples/s]


Step,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
100,0.817405,0.696875,715262.000000,-1.314763,0.139513,1.721232,0.610202,0.155647
200,0.678224,0.653686,1400559.000000,-1.688960,-0.305259,0.888289,0.612539,0.254422
300,0.662867,0.634134,2101685.000000,-1.551073,-0.126817,1.100723,0.657710,0.319424
400,0.655628,0.630820,2800236.000000,-1.804834,-0.326380,0.825392,0.668614,0.345468
500,0.707024,0.618760,3506831.000000,-1.437906,0.088791,1.379728,0.672897,0.401876
600,0.612795,0.609783,4204337.000000,-1.443797,-0.064924,1.110689,0.682243,0.377081
700,0.581161,0.611916,4915145.000000,-1.497047,-0.071818,1.046565,0.671729,0.370330
800,0.674296,0.608911,5616890.000000,-1.322485,-0.058938,0.972364,0.691978,0.341566
900,0.580726,0.601112,6334364.000000,-1.323356,0.163655,1.309178,0.684969,0.417365
1000,0.567207,0.600992,7039402.000000,-1.377921,0.124597,1.300197,0.688474,0.424220


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.51it/s]


TrainOutput(global_step=2016, training_loss=0.6585135721261539, metrics={'train_runtime': 1828.9801, 'train_samples_per_second': 8.815, 'train_steps_per_second': 1.102, 'total_flos': 5.268634303119667e+16, 'train_loss': 0.6585135721261539, 'epoch': 1.0})

In [4]:
trainer.save_model("./checkpoints/rm/final")
tokenizer.save_pretrained("./checkpoints/rm/final")

metrics = trainer.evaluate()
print(f"Eval accuracy: {metrics.get('eval_accuracy', 'N/A')}")
print(f"Eval loss: {metrics.get('eval_loss', 'N/A')}")
print("Model saved to ./checkpoints/rm/final")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.37it/s]


Training Loss,Validation Loss,Step,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
0.681227,0.618760,2016,14249792.000000,-1.437906,0.088791,1.379728,0.672897,0.401876


Eval accuracy: 0.6728971962616822
Eval loss: 0.6187604069709778
Model saved to ./checkpoints/rm/final
